In [1]:
from rdflib import Graph, RDF, RDFS, OWL, Namespace
from rdflib.namespace import SKOS, XSD
import os
from openai import OpenAI

In [2]:
# Read file
with open("GDPR.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Basic stats
print(f"Total characters: {len(text)}")
print(f"Total words: {len(text.split())}")
print(f"Total lines: {text.count(chr(10))}")

Total characters: 345695
Total words: 53937
Total lines: 2469


In [4]:
from rdflib import Graph, RDF, RDFS, OWL, BNode
from rdflib.namespace import SKOS, XSD

# Load CCO ontology
g = Graph()
g.parse("CCO/CCO (V1).ttl", format="turtle")
print("CCO loaded successfully")

def get_restriction_label(g, node):
    on_property = g.value(node, OWL.onProperty)
    prop_label = g.value(on_property, RDFS.label) if on_property else None
    prop_name = str(prop_label) if prop_label else str(on_property).split("#")[-1] if on_property else "?"
    
    min_card = g.value(node, OWL.minQualifiedCardinality)
    max_card = g.value(node, OWL.maxQualifiedCardinality)
    exact_card = g.value(node, OWL.qualifiedCardinality)
    some_values = g.value(node, OWL.someValuesFrom)
    on_class = g.value(node, OWL.onClass)
    on_range = g.value(node, OWL.onDataRange)
    
    target = None
    if on_class:
        if isinstance(on_class, BNode):
            target = "[complex class expression]"
        else:
            target_label = g.value(on_class, RDFS.label)
            target = str(target_label) if target_label else str(on_class).split("#")[-1]
    elif on_range:
        target = str(on_range).split("#")[-1]
    
    if some_values:
        if isinstance(some_values, BNode):
            intersection = g.value(some_values, OWL.intersectionOf)
            if intersection:
                return f"{prop_name} some [RoleHolding AND hasRole some RegulatoryAuthorityRole]"
            return f"{prop_name} some [complex expression]"
        else:
            sv_label = g.value(some_values, RDFS.label)
            sv = str(sv_label) if sv_label else str(some_values).split("#")[-1]
            return f"{prop_name} some {sv}"
    
    if exact_card:
        return f"{prop_name} exactly {exact_card} {target or ''}"
    elif min_card:
        return f"{prop_name} min {min_card} {target or ''}"
    elif max_card:
        return f"{prop_name} max {max_card} {target or ''}"
    
    return f"restriction on {prop_name}"

schema = {"classes": [], "object_properties": [], "datatype_properties": []}

# For schema_str — store constraints per class
class_constraints = {}

print("=" * 50)
print("CLASSES WITH CONSTRAINTS")
print("=" * 50)

for cls in g.subjects(RDF.type, OWL.Class):
    if not str(cls).startswith("http"):
        continue
    
    label = g.value(cls, RDFS.label)
    definition = g.value(cls, SKOS.definition)
    cls_name = str(label) if label else str(cls).split("#")[-1]
    
    constraints = []
    subclasses = []
    
    print(f"\nClass: {cls_name}")
    
    for superclass in g.objects(cls, RDFS.subClassOf):
        if isinstance(superclass, BNode):
            intersection = g.value(superclass, OWL.intersectionOf)
            if intersection:
                print(f"  Constraints (intersection):")
                items = list(g.items(intersection))
                for item in items:
                    if isinstance(item, BNode):
                        restriction = get_restriction_label(g, item)
                        print(f"    AND {restriction}")
                        constraints.append(f"AND {restriction}")
            else:
                restriction = get_restriction_label(g, superclass)
                print(f"  Constraint: {restriction}")
                constraints.append(restriction)
        else:
            superclass_label = g.value(superclass, RDFS.label)
            superclass_name = str(superclass_label) if superclass_label else str(superclass).split("#")[-1]
            print(f"  SubClassOf: {superclass_name}")
            subclasses.append(superclass_name)
    
    if definition:
        print(f"  Definition: {str(definition)[:100]}")
    
    schema["classes"].append(cls_name)
    class_constraints[cls_name] = {
        "subclasses": subclasses,
        "constraints": constraints
    }

print("\n" + "=" * 50)
print("OBJECT PROPERTIES")
print("=" * 50)

for prop in g.subjects(RDF.type, OWL.ObjectProperty):
    label = g.value(prop, RDFS.label)
    domain = g.value(prop, RDFS.domain)
    range_ = g.value(prop, RDFS.range)
    
    prop_name = str(label) if label else str(prop).split("#")[-1]
    domain_name = str(g.value(domain, RDFS.label) or str(domain).split("#")[-1]) if domain else "None"
    range_name = str(g.value(range_, RDFS.label) or str(range_).split("#")[-1]) if range_ else "None"
    
    print(f"\nProperty: {prop_name}")
    print(f"  Domain: {domain_name} → Range: {range_name}")
    
    schema["object_properties"].append({
        "name": prop_name,
        "domain": domain_name,
        "range": range_name
    })

print("\n" + "=" * 50)
print("DATATYPE PROPERTIES")
print("=" * 50)

for prop in g.subjects(RDF.type, OWL.DatatypeProperty):
    label = g.value(prop, RDFS.label)
    domain = g.value(prop, RDFS.domain)
    range_ = g.value(prop, RDFS.range)
    
    prop_name = str(label) if label else str(prop).split("#")[-1]
    domain_name = str(g.value(domain, RDFS.label) or str(domain).split("#")[-1]) if domain else "None"
    range_name = str(range_).split("#")[-1] if range_ else "None"
    
    print(f"\nProperty: {prop_name}")
    print(f"  Domain: {domain_name} → Range: {range_name}")
    
    schema["datatype_properties"].append({
        "name": prop_name,
        "domain": domain_name,
        "range": range_name
    })

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Classes: {len(schema['classes'])}")
print(f"Object Properties: {len(schema['object_properties'])}")
print(f"Datatype Properties: {len(schema['datatype_properties'])}")

# Schema string for LLM — with constraints
schema_str = "CCO CLASSES (with constraints):\n"
for cls_name in schema["classes"]:
    info = class_constraints.get(cls_name, {})
    schema_str += f"- {cls_name}"
    if info.get("subclasses"):
        schema_str += f" (subClassOf: {', '.join(info['subclasses'])})"
    schema_str += "\n"
    for constraint in info.get("constraints", []):
        schema_str += f"    Constraint: {constraint}\n"

schema_str += "\nOBJECT PROPERTIES (Domain → Range):\n"
for prop in schema["object_properties"]:
    schema_str += f"- {prop['name']}: {prop['domain']} → {prop['range']}\n"

schema_str += "\nDATATYPE PROPERTIES (Domain → Range):\n"
for prop in schema["datatype_properties"]:
    schema_str += f"- {prop['name']}: {prop['domain']} → {prop['range']}\n"

print("\n" + "=" * 50)
print("SCHEMA STRING FOR LLM PROMPT")
print("=" * 50)
print(schema_str)

file:///Users/umair/cco-domain%20evaluation/CCO/CCO (V1).ttl does not look like a valid URI, trying to serialize this will break.
file:///Users/umair/cco-domain%20evaluation/CCO/CCO (V1).ttl does not look like a valid URI, trying to serialize this will break.


CCO loaded successfully
CLASSES WITH CONSTRAINTS

Class: Action
  Definition: An action is an activity or behaviour that a norm requires, permits, or prohibits.

Class: Agent
  Definition: An agent is an entity that can perform actions, hold roles, and be subject to regulatory or complian

Class: Condition
  Definition: A condition is a criterion that determines the applicability of a norm or exception.

Class: Exception
  Constraints (intersection):
    AND has condition min 1 Condition
    AND modifies norm min 1 Norm
  Definition: An exception is a normative statement that modifies the applicability or force of a norm in a partic

Class: Norm
  Constraint: has applicability start exactly 1 date
  Constraint: has applicability end max 1 date
  Definition: A norm is a prescriptive rule that specifies what must, may, or must not be done in a regulatory or 

Class: Obligation
  SubClassOf: Norm
  Definition: An obligation is a norm that specifies what must be done in a regulatory or com

In [6]:

os.environ["NSCALE_SERVICE_TOKEN"] = ""
print("Nscale keys set for this notebook session.")

NSCALE_TOKEN = os.environ.get("NSCALE_SERVICE_TOKEN")
NSCALE_BASE_URL = os.environ.get("NSCALE_BASE_URL", "https://inference.api.nscale.com/v1")
MODEL = os.environ.get("NSCALE_MODEL", "Qwen/Qwen3-235B-A22B-Instruct-2507")

if not NSCALE_TOKEN:
    raise RuntimeError("Missing NSCALE_SERVICE_TOKEN env var. Do not paste tokens into notebooks.")

client = OpenAI(api_key=NSCALE_TOKEN, base_url=NSCALE_BASE_URL)

# Quick connection test
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the word OK only"}],
    max_tokens=10,
    temperature=0.0,
)
print("API:", (test.choices[0].message.content or "").strip())
print("Model:", MODEL)


Nscale keys set for this notebook session.
API: OK
Model: Qwen/Qwen3-235B-A22B-Instruct-2507


In [7]:
# Read regulatory text
with open("GDPR.txt", "r", encoding="utf-8") as f:
    reg_text = f.read()

In [8]:
# ============================================================
# HELPER FUNCTIONS — define karo sabse pehle
# ============================================================

import re
import csv
import time
from datetime import datetime
from difflib import SequenceMatcher

def parse_cq_output(text: str, domain_name: str):
    cqs  = []
    gaps = []

    blocks = re.split(r"\n(?=\s*(?:CQ\d+|GAP\d+)\s*:)", text.strip())

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        m_cq = re.match(r"^(CQ\d+)\s*:", block)
        if m_cq:
            cq = {"cq_id": m_cq.group(1), "domain": domain_name}
            q  = re.search(r"Question:\s*(.+?)(?=\nClause:|\Z)",          block, re.DOTALL)
            c  = re.search(r"Clause:\s*(.+?)(?=\nExcerpt:|\Z)",           block, re.DOTALL)
            e  = re.search(r'Excerpt:\s*"?(.+?)"?(?=\nCCO Elements:|\Z)', block, re.DOTALL)
            el = re.search(r"CCO Elements:\s*\[?(.+?)\]?(?=\n[A-Z]|\Z)",  block, re.DOTALL)

            cq["question"]     = q.group(1).strip()  if q  else ""
            cq["clause"]       = c.group(1).strip()  if c  else ""
            cq["excerpt"]      = e.group(1).strip()  if e  else ""
            cq["cco_elements"] = el.group(1).strip() if el else ""
            cq["notes"]        = ""
            cqs.append(cq)
            continue

        m_gap = re.match(r"^(GAP\d+)\s*:", block)
        if m_gap:
            gap = {"gap_id": m_gap.group(1), "domain": domain_name}
            c = re.search(r"Clause:\s*(.+?)(?=\nExcerpt:|\Z)", block, re.DOTALL)
            e = re.search(r'Excerpt:\s*"?(.+?)"?(?=\nReason:|\Z)', block, re.DOTALL)
            r = re.search(r"Reason:\s*(.+?)\Z", block, re.DOTALL)

            gap["clause"]  = c.group(1).strip() if c else ""
            gap["excerpt"] = e.group(1).strip() if e else ""
            gap["reason"]  = r.group(1).strip() if r else ""
            gaps.append(gap)
            continue

    return cqs, gaps

def ensure_ids(items, prefix):
    for idx, item in enumerate(items, start=1):
        item[f"{prefix.lower()}_id"] = f"{prefix}{idx:03d}"
    return items

def chunk_text(text, max_words=2000):
    words   = text.split()
    chunks  = []
    current = []
    for word in words:
        current.append(word)
        if len(current) >= max_words:
            chunks.append(" ".join(current))
            current = []
    if current:
        chunks.append(" ".join(current))
    return chunks

In [10]:
def build_prompt(chunk, chunk_idx, total_chunks, sample_instruction, schema_str):
    return f"""You are a compliance ontology expert.

Below is the schema of the Core Compliance Ontology (CCO). Use ONLY these classes and properties, with their names exactly as given.

{schema_str}

Regulatory text (Data Protection domain — General Data Protection Regulation (GDPR) EU 2016/679, chunk {chunk_idx}/{total_chunks}):

{chunk}

Task: Generate competency questions (CQs) that are directly grounded in the regulatory text and answerable using CCO as a structural model.

Rules:
- {sample_instruction}
- Keep each CQ question concise — one focused question only. Do not combine multiple sub-questions in one CQ.
- Each CQ must be answerable by representing the relevant regulatory clause using CCO classes and properties, and querying it with SPARQL. If not, put it in GAP instead.
- Do NOT propose CQs that would require full-text search. CQs must be answerable via IRIs/structural links created in the ABox, not via string matching on text values.
- Do NOT create CQs about specific data fields, recorded attributes, qualitative assessments, or staff duties not modelled in CCO (e.g., good character, qualifications, competence, health status, risk scoring, gender pay gap). Put these in GAP.
- Do NOT use cco:allocatedTo unless the question is specifically about resource allocation to an agent (Resource → Agent).
- Use cco:appliesUnder for Norm → Condition links. Use cco:hasCondition ONLY for Exception → Condition links.
- "applies from [date]" or "effective from [date]" at regulation level → use cco:Regulation + cco:hasValidityStart (NOT cco:hasApplicabilityStart).
- cco:hasApplicabilityStart is ONLY for norm-level applicability, not regulation-level validity dates.
- Do NOT include both cco:Permission and cco:Obligation in the same CQ — pick one deontic type.
- Do NOT generate CQs from recital paragraphs (numbered in parentheses e.g. (1), (2), (3)). Only generate CQs from Articles.
- Prefer CQs that query: Regulation → Norm (cco:specifiesNorm), deontic type, appliesToRole, temporal validity/applicability, conditions/exceptions, actions/resources.
- Clause references: quote the exact Article number (e.g., "Article 5", "Article 17") and include a short excerpt (<= 25 words).
- Always wrap CCO Elements in square brackets: CCO Elements: [cco:Obligation, cco:appliesToRole, ...]

For each CQ provide:
Question: ...
Clause: ...
Excerpt: "..."
CCO Elements: [comma-separated list like cco:Obligation, cco:appliesToRole, cco:hasAction ...]

For each GAP provide:
Clause: ...
Excerpt: "..."
Reason: ... (be specific)

Output format strictly:
CQ1:
Question: ...
Clause: ...
Excerpt: "..."
CCO Elements: [...]

GAP1:
Clause: ...
Excerpt: "..."
Reason: ...
"""

# ============================================================
# Config
# ============================================================
SAMPLE_SIZE  = None
DOMAIN       = "Data Protection"
CHUNK_WORDS  = 2000
MAX_RETRIES  = 3
TIMEOUT_SECS = 300

RAW_OUTPUT_FILE = "raw_output_gdpr.txt"
cq_csv          = "cqs_gdpr.csv"
gap_csv         = "gaps_gdpr.csv"

sample_instruction = (
    f"Generate exactly {SAMPLE_SIZE} CQs as a sample for review."
    if SAMPLE_SIZE else
    "Generate all possible CQs from this regulatory text chunk."
)

# ============================================================
# STEP 3 — Chunked LLM Call
# ============================================================
chunks       = chunk_text(reg_text, max_words=CHUNK_WORDS)
all_outputs  = []
all_cqs_raw  = []
all_gaps_raw = []

print(f"Regulatory text split into {len(chunks)} chunks")
print(f"Processing...\n")

total_start = datetime.now()

for chunk_idx, chunk in enumerate(chunks, start=1):
    print(f"{'='*50}")
    print(f"Chunk {chunk_idx}/{len(chunks)} — {len(chunk.split())} words")
    print(f"{'='*50}")

    prompt       = build_prompt(chunk, chunk_idx, len(chunks), sample_instruction, schema_str)
    chunk_output = ""
    success      = False

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"  Attempt {attempt}/{MAX_RETRIES}...")
            chunk_start = datetime.now()

            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=4000,
                temperature=0.0,
                timeout=TIMEOUT_SECS,
            )

            chunk_output = response.choices[0].message.content or ""
            elapsed      = (datetime.now() - chunk_start).seconds
            print(f"   Done in {elapsed}s — {len(chunk_output)} chars")
            success = True
            break

        except Exception as e:
            print(f"   Attempt {attempt} failed: {type(e).__name__}: {e}")
            if attempt < MAX_RETRIES:
                wait = 10 * attempt
                print(f"  Waiting {wait}s before retry...")
                time.sleep(wait)
            else:
                print(f"    Chunk {chunk_idx} skipped after {MAX_RETRIES} attempts")

    if not success or not chunk_output:
        continue

    # Save raw output per chunk
    raw_file = f"raw_output_gdpr_chunk{chunk_idx}.txt"
    with open(raw_file, "w", encoding="utf-8") as f:
        f.write(chunk_output)
    all_outputs.append(chunk_output)

    # Parse chunk
    chunk_cqs, chunk_gaps = parse_cq_output(chunk_output, DOMAIN)
    all_cqs_raw.extend(chunk_cqs)
    all_gaps_raw.extend(chunk_gaps)

    print(f"  CQs: {len(chunk_cqs)} | GAPs: {len(chunk_gaps)} | "
          f"Total so far: {len(all_cqs_raw)} CQs, {len(all_gaps_raw)} GAPs")

    # Incremental save after each chunk
    with open(cq_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["cq_id","domain","clause","excerpt","question","cco_elements","notes"],
            extrasaction="ignore",
        )
        writer.writeheader()
        for cq in all_cqs_raw:
            cq.setdefault("notes","")
            writer.writerow(cq)

    with open(gap_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["gap_id","domain","clause","excerpt","reason"],
            extrasaction="ignore",
        )
        writer.writeheader()
        for gap in all_gaps_raw:
            writer.writerow(gap)

    print(f"   Progress saved → {cq_csv}, {gap_csv}")

# Save combined raw output
with open(RAW_OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_outputs))

total_elapsed = (datetime.now() - total_start).seconds
print(f"\n{'='*50}")
print(f"GENERATION COMPLETE")
print(f"{'='*50}")
print(f"Total time:  {total_elapsed}s")
print(f"Total CQs:   {len(all_cqs_raw)}")
print(f"Total GAPs:  {len(all_gaps_raw)}")


# ============================================================
# STEP 4 — Re-number IDs globally
# ============================================================
cqs  = ensure_ids(all_cqs_raw,  "CQ")
gaps = ensure_ids(all_gaps_raw, "GAP")

print(f"\nAfter ID re-numbering:")
print(f"CQs:  {len(cqs)}")
print(f"GAPs: {len(gaps)}")


# ============================================================
# STEP 5 — Deduplicate + Auto-flag + Auto-fix
# ============================================================
def normalize(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def elements_set(el_str: str):
    return set(x.strip() for x in (el_str or "").split(",") if x.strip())

def dedupe_cqs(cqs, similarity_threshold=0.80):
    for cq in cqs:
        cq["keep"]      = True
        cq["dup_of"]    = ""
        cq["dup_score"] = ""
        cq["dup_stage"] = ""

    for i, cq_i in enumerate(cqs):
        if not cq_i.get("keep"):
            continue
        q_i      = normalize(cq_i.get("question", ""))
        dom_i    = cq_i.get("domain", "")
        clause_i = str(cq_i.get("clause", "")).strip()

        for j in range(i + 1, len(cqs)):
            cq_j = cqs[j]
            if not cq_j.get("keep"):
                continue
            q_j      = normalize(cq_j.get("question", ""))
            dom_j    = cq_j.get("domain", "")
            clause_j = str(cq_j.get("clause", "")).strip()

            if dom_i == dom_j and q_i and (q_i == q_j):
                cq_j["keep"]      = False
                cq_j["dup_of"]    = cq_i["cq_id"]
                cq_j["dup_score"] = "1.00"
                cq_j["dup_stage"] = "A-exact"
                continue

            if dom_i == dom_j and clause_i == clause_j and q_i and q_j:
                score = SequenceMatcher(None, q_i, q_j).ratio()
                if score >= similarity_threshold:
                    cq_j["keep"]      = False
                    cq_j["dup_of"]    = cq_i["cq_id"]
                    cq_j["dup_score"] = f"{score:.2f}"
                    cq_j["dup_stage"] = "B-similar"

    return cqs

def fix_cco_elements(cq):
    q   = (cq.get("question", "") or "").lower()
    els = elements_set(cq.get("cco_elements", ""))

    if ("apply from" in q or "effective from" in q or "start date" in q) and (
        "guideline" in q or "guidelines" in q or "regulation" in q or "updated" in q
    ):
        els.discard("cco:hasApplicabilityStart")
        els.discard("cco:hasApplicabilityEnd")
        els.add("cco:Regulation")
        els.add("cco:hasValidityStart")

    cq["needs_human_fix"] = False
    if "cco:hasCondition" in els and "cco:Exception" not in els:
        cq["needs_human_fix"] = True

    cq["cco_elements"] = ", ".join(sorted(els))

def auto_flag(cq):
    q   = (cq.get("question", "") or "").lower()
    els = elements_set(cq.get("cco_elements", ""))

    bad = False
    if "cco:regulates" in els:
        bad = True
    if "cco:hasApplicabilityStart" in els and (
        "guidelines apply" in q or "updated guidelines" in q or
        "start date" in q or "application of the guidelines" in q
    ):
        bad = True
    if "cco:hasCondition" in els and "cco:Exception" not in els:
        bad = True
    if "cco:Permission" in els and "cco:Obligation" in els:
        bad = True
    if "cco:Resource" in els and ("committee" in q or "function" in q or "role" in q):
        bad = True

    cq["needs_human_fix"] = cq.get("needs_human_fix", False) or bad

cqs_with_flags = dedupe_cqs(cqs, similarity_threshold=0.80)
for cq in cqs_with_flags:
    fix_cco_elements(cq)
    auto_flag(cq)

keep_count      = sum(1 for cq in cqs_with_flags if cq.get("keep"))
flag_count      = sum(1 for cq in cqs_with_flags if not cq.get("keep"))
needs_fix_count = sum(1 for cq in cqs_with_flags if cq.get("keep") and cq.get("needs_human_fix"))

print(f"\n{'='*50}")
print("DEDUPLICATION SUMMARY")
print(f"{'='*50}")
print(f"Total CQs:              {len(cqs_with_flags)}")
print(f"Keep (True):            {keep_count}")
print(f"Flagged duplicates:     {flag_count}")
print(f"Needs human fix (keep): {needs_fix_count}")
print(f"GAPs:                   {len(gaps)}")


# ============================================================
# STEP 6 — Final Save for Human Curation
# ============================================================
with open(cq_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "cq_id","domain","clause","excerpt","question","cco_elements",
            "keep","dup_of","dup_score","dup_stage",
            "needs_human_fix",
            "human_keep","human_fix_question","human_fix_elements","human_notes",
            "notes",
        ],
        extrasaction="ignore",
    )
    writer.writeheader()
    for cq in cqs_with_flags:
        cq.setdefault("needs_human_fix",    False)
        cq.setdefault("human_keep",         "")
        cq.setdefault("human_fix_question", "")
        cq.setdefault("human_fix_elements", "")
        cq.setdefault("human_notes",        "")
        cq.setdefault("notes",              "")
        writer.writerow(cq)

with open(gap_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["gap_id","domain","clause","excerpt","reason",
                    "human_confirm_gap","human_notes"],
        extrasaction="ignore",
    )
    writer.writeheader()
    for gap in gaps:
        gap.setdefault("human_confirm_gap", "")
        gap.setdefault("human_notes",       "")
        writer.writerow(gap)

print(f"\n{'='*50}")
print("FINAL SUMMARY — Data Protection Domain (GDPR)")
print(f"{'='*50}")
print(f"Total CQs:              {len(cqs_with_flags)}")
print(f"Keep (True):            {keep_count}")
print(f"Flagged duplicates:     {flag_count}")
print(f"Needs human fix:        {needs_fix_count}  ← review these first")
print(f"Total GAPs:             {len(gaps)}")
print(f"CQs saved →             {cq_csv}")
print(f"GAPs saved →            {gap_csv}")
print(f"Raw output saved →      {RAW_OUTPUT_FILE}")

Regulatory text split into 27 chunks
Processing...

Chunk 1/27 — 2000 words
  Attempt 1/3...
  ✅ Done in 68s — 9496 chars
  CQs: 20 | GAPs: 6 | Total so far: 20 CQs, 6 GAPs
  💾 Progress saved → cqs_gdpr.csv, gaps_gdpr.csv
Chunk 2/27 — 2000 words
  Attempt 1/3...
  ✅ Done in 66s — 9232 chars
  CQs: 19 | GAPs: 5 | Total so far: 39 CQs, 11 GAPs
  💾 Progress saved → cqs_gdpr.csv, gaps_gdpr.csv
Chunk 3/27 — 2000 words
  Attempt 1/3...
  ✅ Done in 53s — 7179 chars
  CQs: 15 | GAPs: 6 | Total so far: 54 CQs, 17 GAPs
  💾 Progress saved → cqs_gdpr.csv, gaps_gdpr.csv
Chunk 4/27 — 2000 words
  Attempt 1/3...
  ✅ Done in 96s — 14013 chars
  CQs: 25 | GAPs: 11 | Total so far: 79 CQs, 28 GAPs
  💾 Progress saved → cqs_gdpr.csv, gaps_gdpr.csv
Chunk 5/27 — 2000 words
  Attempt 1/3...
  ✅ Done in 67s — 8826 chars
  CQs: 20 | GAPs: 7 | Total so far: 99 CQs, 35 GAPs
  💾 Progress saved → cqs_gdpr.csv, gaps_gdpr.csv
Chunk 6/27 — 2000 words
  Attempt 1/3...
  ✅ Done in 78s — 11491 chars
  CQs: 20 | GAPs: 8 |